going to fine tune the LLM with the best hyperparameter (from fine tune LLM on traintest 5)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import pandas as pd
from datasets import Dataset

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
print(device)

cuda


In [6]:
file_path_train = '/content/drive/My Drive/LLM/trainDataset.csv'

In [7]:
df = pd.read_csv(file_path_train)

In [8]:
df.head()

,sequence
0,"android.net.nsd.STATE_CHANGED, android.net.con..."
1,"android.net.nsd.STATE_CHANGED, android.net.con..."
2,"android.intent.action.DREAMING_STARTED, androi..."
3,"android.net.wifi.RSSI_CHANGED, android.intent...."
4,"android.intent.action.DREAMING_STARTED, androi..."


In [9]:
sequence_list = df['sequence'].tolist()

In [12]:
# len(sequence_list)

5000

In [ ]:
trainDataset = []
for sequence in sequence_list:
    result = sequence.split(",")
    result = [item.strip() for item in result]
    trainDataset.append(result)

In [ ]:
print(trainDataset[0])

['android.net.nsd.STATE_CHANGED', 'android.net.conn.CONNECTIVITY_CHANGE', 'android.net.wifi.STATE_CHANGE', 'android.net.wifi.WIFI_STATE_CHANGED', 'android.net.wifi.p2p.STATE_CHANGED', 'android.net.wifi.supplicant.STATE_CHANGE', 'android.media.ACTION_SCO_AUDIO_STATE_UPDATED', 'android.media.RINGER_MODE_CHANGED', 'android.media.SCO_AUDIO_STATE_CHANGED']


In [ ]:
# trainDataset = trainDataset[:1000]

In [ ]:
trainDataset2 = []
for sequence in trainDataset:
    input = " ".join(sequence[:-1])
    row = [input, sequence[-1]]
    trainDataset2.append(row)

In [ ]:
# Step 1: Split into two lists
inputs = [pair[0] for pair in trainDataset2]
targets = [pair[1] for pair in trainDataset2]

In [ ]:
data_dict = {
    "input": inputs,
    "target": targets
}

In [ ]:
dataset = Dataset.from_dict(data_dict)

In [ ]:
print(dataset)

Dataset({
    features: ['input', 'target'],
    num_rows: 5000
})


In [ ]:
import torch
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
from datasets import Dataset, DatasetDict
import random
from unsloth.chat_templates import get_chat_template

/tmp/ipython-input-3279917189.py:4: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth.chat_templates import get_chat_template


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# %pip install -qqq trl transformers accelerate peft bitsandbytes wandb datasets unsloth --upgrade

In [ ]:
%%capture
%pip install -U transformers
%pip install -U datasets
%pip install -U accelerate
%pip install -U peft
%pip install -U trl
%pip install -U bitsandbytes
%pip install -U wandb

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)
import os, torch, wandb
from datasets import load_dataset
from trl import SFTTrainer, setup_chat_format

In [ ]:
base_model = "unsloth/Llama-3.2-1B-bnb-4bit"
new_model = "llama-3.2-3b-it-Ecommerce-ChatBot"

In [ ]:
# Set torch dtype and attention implementation
if torch.cuda.get_device_capability()[0] >= 8:
    # !pip install -qqq flash-attn
    torch_dtype = torch.bfloat16
    # attn_implementation = "flash_attention_2"
else:
    torch_dtype = torch.float16
    # attn_implementation = "eager"

In [ ]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)
# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    # attn_implementation=attn_implementation
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

In [ ]:
print(dataset)

Dataset({
    features: ['input', 'target'],
    num_rows: 5000
})


In [ ]:
dataset = dataset.shuffle(seed=65).select(range(5000))

In [ ]:
print(dataset)

Dataset({
    features: ['input', 'target'],
    num_rows: 5000
})


In [ ]:
instruction = """You are a Smartphone Event Predictor agent named John.
    Analyze the input and predict the target.
    """

# Set the chat template manually
tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|begin_of_text|>' + message['role'] + '\n' + message['content'] + '<|end_of_text|>\n'}}{% endfor %}{% if add_generation_prompt %}{{'<|begin_of_text|>assistant\n'}}{% endif %}"


def format_chat_template(row):

    row_json = [{"role": "system", "content": instruction },
               {"role": "user", "content": row["input"]},
               {"role": "assistant", "content": row["target"]}]

    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

dataset = dataset.map(
    format_chat_template,
    # Removed num_proc=4 for now to simplify debugging
    # num_proc= 4,
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['input', 'target', 'text'],
    num_rows: 5000
})

In [ ]:
import bitsandbytes as bnb

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

modules = find_all_linear_names(model)

In [ ]:
# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules
)
model = get_peft_model(model, peft_config)

In [ ]:
#Hyperparamter
training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=2,
    eval_strategy="no",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=50,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    group_by_length=True,
    report_to="wandb"
)

In [ ]:
# Setting sft parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset, # Use the entire dataset for training
    # eval_dataset=dataset["test"], # Remove eval_dataset as it's not available
    peft_config=peft_config,
    max_seq_length= 512,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_arguments,
    packing= False,
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: khanamit23 (khanamit23-university-of-north-texas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,4.169900
2,5.011800
3,4.979000
4,5.035600
5,5.113700
6,4.976900
7,4.951800
8,4.729100
9,4.895700
10,4.342000


TrainOutput(global_step=5000, training_loss=0.12400458695494454, metrics={'train_runtime': 2828.7232, 'train_samples_per_second': 3.535, 'train_steps_per_second': 1.768, 'total_flos': 3209550186283008.0, 'train_loss': 0.12400458695494454, 'epoch': 2.0})

In [ ]:
file_path_test = '/content/drive/My Drive/LLM/testDataset.csv'

In [ ]:
dfTest = pd.read_csv(file_path_test)

In [ ]:
dfTest.head()

,sequence
0,"android.intent.action.SCREEN_OFF, android.net...."
1,"android.net.conn.CONNECTIVITY_CHANGE, android...."
2,"android.net.wifi.WIFI_STATE_CHANGED, android.n..."
3,"android.net.wifi.supplicant.STATE_CHANGE, andr..."
4,"android.net.wifi.supplicant.STATE_CHANGE, andr..."


In [ ]:
sequence_list_test = dfTest['sequence'].tolist()

In [ ]:
testDataset = []
for sequence in sequence_list_test:
    result = sequence.split(",")
    result = [item.strip() for item in result]
    testDataset.append(result)

In [ ]:
# testDataset = testDataset[:100]

In [ ]:
testDataset2 = []
for sequence in testDataset:
    input = " ".join(sequence[:-1])
    row = [input, sequence[-1]]
    testDataset2.append(row)

In [ ]:
# Step 1: Split into two lists
inputsTest = [pair[0] for pair in testDataset2]
targetsTest = [pair[1] for pair in testDataset2]

In [ ]:
data_dict_test = {
    "input": inputsTest,
    "target": targetsTest
}

In [ ]:
datasetTest = Dataset.from_dict(data_dict_test)

In [ ]:
print(datasetTest)

Dataset({
    features: ['input', 'target'],
    num_rows: 2000
})


In [ ]:
print(datasetTest['input'][0])
print(datasetTest['input'][99])

android.intent.action.SCREEN_OFF
android.intent.action.DREAMING_STOPPED


In [ ]:
predictedTagetsTest = []

for i in range(2000):
    messages = [{"role": "system", "content": instruction},
      {"role": "user", "content": datasetTest['input'][i]}]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=150, num_return_sequences=1)

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    predictedEvent = text.split("assistant")[1].lstrip("\n")

    predictedTagetsTest.append(predictedEvent)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
precision = precision_score(targetsTest, predictedTagetsTest, average='micro')
recall = recall_score(targetsTest, predictedTagetsTest, average='micro')
f1_weighted = f1_score(targetsTest, predictedTagetsTest, average='weighted')
f1_macro = f1_score(targetsTest, predictedTagetsTest, average='macro')
print("Precision:", precision)
print("Recall:", recall)
print("F1-weighted:", f1_weighted)
print("F1-macro:", f1_macro)

Precision: 0.559
Recall: 0.559
F1-weighted: 0.5788707958094825
F1-macro: 0.21259247438473747
